In [9]:
import os
import os.path as op
import numpy as np
import pandas as pd
import json
import pickle
# import matplotlib.pyplot as plt
# from scipy.ndimage import label

In [11]:
json_file = "settings.json"
with open(json_file) as pipeline_file:
    parameters = json.load(pipeline_file)
path = parameters["dataset_path"]

der_path = op.join(path, "derivatives_v2")
proc_path = op.join(der_path, "processed")

In [5]:
epoch_types=['STIM','RESP']
condition_names=['SHORT','LONG']

In [6]:
td_color=np.array([27,158,119])/255.0
asd_color=np.array([117,112,179])/255.0

In [44]:
subj_dfs=[]

df = pd.read_csv(op.join(path, 'data_v2', 'GOGO_Demographics_2025_COMO.csv'))
# Collect subject tuples first to keep the main body clean for joblib
subjects = list(df.loc[:, ["ParticipantID", "Status"]].itertuples(index=False, name=None))
for subject_id, group in subjects:
    if group=='TD':
        sub_path = op.join(proc_path, subject_id)

        behav_path = op.join(sub_path, f'autoreject-{subject_id}-epo-behav.csv')
        if op.exists(behav_path):
            behav_df = pd.read_csv(behav_path)
            
            n_long=len(np.where(behav_df['condition']=='LONG')[0])
            new_trial_idx=np.arange(n_long)
            behav_df.loc[behav_df['condition']=='LONG', 'trial_idx']=new_trial_idx
            
            n_short=len(np.where(behav_df['condition']=='SHORT')[0])
            new_trial_idx=np.arange(n_short)
            behav_df.loc[behav_df['condition']=='SHORT', 'trial_idx']=new_trial_idx

            subj_dfs.append(behav_df)

df = pd.read_csv(op.join(path, 'data_v2', 'GOGO_Demographics_2025_Driving.csv'))
# Collect subject tuples first to keep the main body clean for joblib
subjects = list(df.loc[:, ["ParticipantID", "Status"]].itertuples(index=False, name=None))
for subject_id, group in subjects:
    if group=='TD':
        sub_path = op.join(proc_path, subject_id)

        behav_path = op.join(sub_path, f'autoreject-{subject_id}-epo-behav.csv')
        if op.exists(behav_path):
            behav_df = pd.read_csv(behav_path)
            
            n_long=len(np.where(behav_df['condition']=='LONG')[0])
            new_trial_idx=np.arange(n_long)
            behav_df.loc[behav_df['condition']=='LONG', 'trial_idx']=new_trial_idx
            
            n_short=len(np.where(behav_df['condition']=='SHORT')[0])
            new_trial_idx=np.arange(n_short)
            behav_df.loc[behav_df['condition']=='SHORT', 'trial_idx']=new_trial_idx
            
            subj_dfs.append(behav_df)
all_subj_df=pd.concat(subj_dfs)

In [45]:
all_subj_df

,subject_id,trial_idx,condition,response_time,stim_kept,resp_kept
0,COM032,0,LONG,0.331667,True,True
1,COM032,1,LONG,0.318333,True,True
2,COM032,2,LONG,0.281667,True,True
3,COM032,0,SHORT,0.468333,True,True
4,COM032,3,LONG,0.316667,True,True
...,...,...,...,...,...,...
152,D083,116,LONG,0.248333,True,True
153,D083,36,SHORT,0.301667,True,True
154,D083,117,LONG,0.283333,True,True
155,D083,118,LONG,0.235000,True,True


In [46]:
burst_df = pd.read_csv('/home/common/bonaiuto/gogo_bursts/derivatives_v2/processed/output/TD_contra_burst_features.csv')
burst_df = burst_df.rename(columns={'trial': 'trial_idx'})

In [47]:
burst_df

,subject_id,group,epoch_type,condition,cluster,channel,trial_idx,peak_freq,peak_amp_iter,peak_amp_base,...,PC_11,PC_12,PC_13,PC_14,PC_15,PC_16,PC_17,PC_18,PC_19,PC_20
0,COM032,TD,STIM,SHORT,contra,MLC21,0.0,20.584034,4.724776e-14,4.724776e-14,...,0.780050,0.498702,-0.206223,-1.074867,0.689881,0.333411,0.044097,-0.990487,0.046062,0.128317
1,COM032,TD,STIM,SHORT,contra,MLC21,0.0,24.600840,3.643034e-14,3.643034e-14,...,-0.457894,-0.195784,-0.450063,0.492474,0.200955,0.882033,1.067123,-0.101616,0.242095,0.094088
2,COM032,TD,STIM,SHORT,contra,MLC21,0.0,21.588235,2.970509e-14,2.971232e-14,...,-0.296921,-1.229713,0.578595,0.539911,-0.189090,-0.334086,-0.090645,-0.269638,-0.014656,-0.239522
3,COM032,TD,STIM,SHORT,contra,MLC21,0.0,20.584034,2.872534e-14,3.506656e-14,...,0.019113,0.030048,-1.047571,0.886113,-1.506943,-0.161024,0.447867,-0.073188,-0.462727,-0.027299
4,COM032,TD,STIM,SHORT,contra,MLC21,0.0,21.588235,2.159214e-14,2.522343e-14,...,0.430503,-0.951223,0.228254,-0.434195,-0.434515,0.511731,0.284670,-0.969984,-0.344390,-0.261792
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1139909,D083,TD,RESP,LONG,contra,MLC62,119.0,18.575630,1.178547e-14,1.478536e-14,...,-1.579125,-0.681705,-0.301526,-0.758289,-0.287087,-0.123636,-0.166603,-1.139740,0.574046,1.244933
1139910,D083,TD,RESP,LONG,contra,MLC62,119.0,25.605042,1.148910e-14,1.309800e-14,...,-0.578059,0.591276,1.482835,0.577815,0.513495,-0.990819,-0.427739,0.094559,-0.064122,0.407439
1139911,D083,TD,RESP,LONG,contra,MLC62,119.0,24.600840,1.137801e-14,1.138651e-14,...,0.259220,1.239013,-1.023419,0.330956,0.473344,-1.461808,0.630572,0.431694,0.807421,0.892128
1139912,D083,TD,RESP,LONG,contra,MLC62,119.0,15.563025,1.106642e-14,1.122690e-14,...,-0.038428,-0.951995,-0.000763,-0.972845,-1.471146,0.195568,0.481486,-0.702310,0.140304,-0.520450


In [48]:
lookup_df = all_subj_df[["subject_id", "trial_idx", "condition", "response_time"]]

# Merge the two DataFrames based on the specified columns and fill missing values with NaN
df_burst_behav = burst_df.merge(lookup_df, on=["subject_id", "trial_idx", "condition"], how='left')


In [49]:
df_burst_behav

,subject_id,group,epoch_type,condition,cluster,channel,trial_idx,peak_freq,peak_amp_iter,peak_amp_base,...,PC_12,PC_13,PC_14,PC_15,PC_16,PC_17,PC_18,PC_19,PC_20,response_time
0,COM032,TD,STIM,SHORT,contra,MLC21,0.0,20.584034,4.724776e-14,4.724776e-14,...,0.498702,-0.206223,-1.074867,0.689881,0.333411,0.044097,-0.990487,0.046062,0.128317,0.468333
1,COM032,TD,STIM,SHORT,contra,MLC21,0.0,24.600840,3.643034e-14,3.643034e-14,...,-0.195784,-0.450063,0.492474,0.200955,0.882033,1.067123,-0.101616,0.242095,0.094088,0.468333
2,COM032,TD,STIM,SHORT,contra,MLC21,0.0,21.588235,2.970509e-14,2.971232e-14,...,-1.229713,0.578595,0.539911,-0.189090,-0.334086,-0.090645,-0.269638,-0.014656,-0.239522,0.468333
3,COM032,TD,STIM,SHORT,contra,MLC21,0.0,20.584034,2.872534e-14,3.506656e-14,...,0.030048,-1.047571,0.886113,-1.506943,-0.161024,0.447867,-0.073188,-0.462727,-0.027299,0.468333
4,COM032,TD,STIM,SHORT,contra,MLC21,0.0,21.588235,2.159214e-14,2.522343e-14,...,-0.951223,0.228254,-0.434195,-0.434515,0.511731,0.284670,-0.969984,-0.344390,-0.261792,0.468333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1139909,D083,TD,RESP,LONG,contra,MLC62,119.0,18.575630,1.178547e-14,1.478536e-14,...,-0.681705,-0.301526,-0.758289,-0.287087,-0.123636,-0.166603,-1.139740,0.574046,1.244933,0.216667
1139910,D083,TD,RESP,LONG,contra,MLC62,119.0,25.605042,1.148910e-14,1.309800e-14,...,0.591276,1.482835,0.577815,0.513495,-0.990819,-0.427739,0.094559,-0.064122,0.407439,0.216667
1139911,D083,TD,RESP,LONG,contra,MLC62,119.0,24.600840,1.137801e-14,1.138651e-14,...,1.239013,-1.023419,0.330956,0.473344,-1.461808,0.630572,0.431694,0.807421,0.892128,0.216667
1139912,D083,TD,RESP,LONG,contra,MLC62,119.0,15.563025,1.106642e-14,1.122690e-14,...,-0.951995,-0.000763,-0.972845,-1.471146,0.195568,0.481486,-0.702310,0.140304,-0.520450,0.216667


In [50]:
stim_epoch_lims=(-1.5, .5)
resp_epoch_lims=(-.5, 1.5)

In [57]:
def compute_burst_counts(
    df_burst_behav, epoch,
    window_width=0.2, step_size=0.025,
    epoch_lims=(-1.0, 1.5)
):
    output_dir = os.path.join(proc_path, 'output')
    os.makedirs(output_dir, exist_ok=True)

    time_centers = np.arange(epoch_lims[0] + window_width / 2,
                             epoch_lims[1] - window_width / 2 + step_size,
                             step_size)
    time_columns = [f"time_{round(tc, 3)}" for tc in time_centers]

    pcs = [col for col in df_burst_behav.columns if col.startswith("PC_")]
    burst_feature_cols = ['peak_time', 'peak_freq', 'peak_amp_base', 'fwhm_freq', 'fwhm_time', 'peak_amp_iter', 'peak_adjustment', 'polarity', 'cluster', 'channel']
    metadata_cols = [col for col in df_burst_behav.columns if col not in pcs + burst_feature_cols]
    group_cols = ["subject_id", "trial_idx", "condition"]

    trial_groups = df_burst_behav.groupby(group_cols)
    trial_records = []

    for (subject, trial, condition), trial_df in trial_groups:
        trial_meta = trial_df.iloc[0][metadata_cols].to_dict()
        
        # Count bursts in time windows
        peak_times = trial_df["peak_time"].values
        for tc, col in zip(time_centers, time_columns):
            count = np.sum((peak_times >= tc - window_width / 2) & (peak_times < tc + window_width / 2))
            trial_meta[col] = count

        trial_records.append(trial_meta)

    df = pd.DataFrame(trial_records)
    df.to_csv(os.path.join(output_dir, f"overall_{epoch}_trial_burst_counts.csv"), index=False)


def compute_burst_counts_by_pc_quartile(
    df_burst_behav, epoch,
    window_width=0.2, step_size=0.025,
    epoch_lims=(-1.0, 1.5), n_q=3
):
    output_dir = './output'
    output_dir = os.path.join(proc_path, 'output')

    time_centers = np.arange(epoch_lims[0] + window_width / 2,
                             epoch_lims[1] - window_width / 2 + step_size,
                             step_size)
    time_columns = [f"time_{round(tc, 3)}" for tc in time_centers]

    pcs = [col for col in df_burst_behav.columns if col.startswith("PC_")]
    burst_feature_cols = ['peak_time', 'peak_freq', 'peak_amp_base', 'fwhm_freq', 'fwhm_time', 'peak_amp_iter', 'peak_adjustment', 'polarity', 'cluster', 'channel']
    metadata_cols = [col for col in df_burst_behav.columns if col not in pcs + burst_feature_cols]
    group_cols = ["subject_id", "trial_idx", "condition"]
    
    for pc in pcs:
        print(f"Processing {pc}")
        step = 100 / n_q
        q_bins = np.percentile(df_burst_behav[pc], np.arange(0, 100 + step, step))
        quartile_dfs = []

        for q in range(n_q):
            df_q = df_burst_behav[
                (df_burst_behav[pc] >= q_bins[q]) &
                (df_burst_behav[pc] < q_bins[q + 1])
            ].copy()
            df_q["tertile"] = q + 1

            trial_groups = df_q.groupby(group_cols)
            trial_records = []

            for (subject, trial, condition), trial_df in trial_groups:
                trial_meta = trial_df.iloc[0][metadata_cols].to_dict()
                trial_meta["tertile"] = q + 1

                # Count bursts in time windows
                peak_times = trial_df["peak_time"].values
                for tc, col in zip(time_centers, time_columns):
                    count = np.sum((peak_times >= tc - window_width / 2) & (peak_times < tc + window_width / 2))
                    trial_meta[col] = count

                trial_records.append(trial_meta)

            quartile_df = pd.DataFrame(trial_records)
            quartile_dfs.append(quartile_df)

        pc_df = pd.concat(quartile_dfs, ignore_index=True)
        pc_df.to_csv(os.path.join(output_dir, f"{pc}_{epoch}_trial_burst_counts.csv"), index=False)


In [55]:
df_stim = df_burst_behav[df_burst_behav['epoch_type'] == 'STIM']
df_resp = df_burst_behav[df_burst_behav['epoch_type'] == 'RESP']

In [58]:
compute_burst_counts(df_stim, 'STIM', epoch_lims=stim_epoch_lims)
compute_burst_counts(df_resp, 'RESP', epoch_lims=resp_epoch_lims)

In [59]:
compute_burst_counts_by_pc_quartile(df_stim, 'STIM', epoch_lims=stim_epoch_lims)
compute_burst_counts_by_pc_quartile(df_resp, 'RESP', epoch_lims=resp_epoch_lims)

Processing PC_1
Processing PC_2
Processing PC_3
Processing PC_4
Processing PC_5
Processing PC_6
Processing PC_7
Processing PC_8
Processing PC_9
Processing PC_10
Processing PC_11
Processing PC_12
Processing PC_13
Processing PC_14
Processing PC_15
Processing PC_16
Processing PC_17
Processing PC_18
Processing PC_19
Processing PC_20
Processing PC_1
Processing PC_2
Processing PC_3
Processing PC_4
Processing PC_5
Processing PC_6
Processing PC_7
Processing PC_8
Processing PC_9
Processing PC_10
Processing PC_11
Processing PC_12
Processing PC_13
Processing PC_14
Processing PC_15
Processing PC_16
Processing PC_17
Processing PC_18
Processing PC_19
Processing PC_20
